In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent 
sys.path.append(str(project_root))

In [2]:
from services.api import *
from services.apitopandas import *
from services.functions import *
import pandas as pd

# Exploring response from api function and the structure of the json

Information to use the api function 
Indicator: which dataset u want to return ( "GDP" or "DEFICIT")
Country: choose country in 2-3 Codes ex. GR for Greece, GE for Germany



In [3]:
raw_data_HICP=ecb_api("HICP","GR")

In [4]:
df_HICP=api_to_PandasDataframe("HICP","GR")
df_HICP.head()

,period,HICP
0,1996-12,NaN
1,1997-01,6.6
2,1997-02,6.5
3,1997-03,5.9
4,1997-04,5.7


In [5]:
df_HICP=clean_df(df_HICP,"HICP")
df_HICP

,HICP,year,month,quartal,period
0,NaN,1996,12,Q4,1996-Q4
1,6.6,1997,1,Q1,1997-Q1
2,6.5,1997,2,Q1,1997-Q1
3,5.9,1997,3,Q1,1997-Q1
4,5.7,1997,4,Q2,1997-Q2
...,...,...,...,...,...
349,2.9,2026,1,Q1,2026-Q1
350,3.1,2026,2,Q1,2026-Q1
351,3.4,2026,3,Q1,2026-Q1
352,4.6,2026,4,Q2,2026-Q2


In [6]:
df_HICP.info()

<class 'pandas.DataFrame'>
RangeIndex: 354 entries, 0 to 353
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   HICP     353 non-null    float64
 1   year     354 non-null    str    
 2   month    354 non-null    int64  
 3   quartal  354 non-null    str    
 4   period   354 non-null    str    
dtypes: float64(1), int64(1), str(3)
memory usage: 18.5 KB


In [7]:

null_period=df_HICP[df_HICP['HICP'].isnull()]

In [8]:
null_period

,HICP,year,month,quartal,period
0,NaN,1996,12,Q4,1996-Q4


In [9]:

df_GDP=api_to_PandasDataframe("GDP","GR")

In [10]:
df_GDP=clean_df(df_GDP,"GDP")
df_GDP=growth_rate(df_GDP,"GDP",1)
df_GDP

,period,GDP,year,quartal,GDP_growth
0,1995-Q1,34893.038761,1995,Q1,NaN
1,1995-Q2,35826.213703,1995,Q2,2.674387
2,1995-Q3,39119.008421,1995,Q3,9.191021
3,1995-Q4,40151.333263,1995,Q4,2.638934
4,1996-Q1,35704.489505,1996,Q1,-11.075208
...,...,...,...,...,...
120,2025-Q1,45863.170264,2025,Q1,-10.594605
121,2025-Q2,50698.430561,2025,Q2,10.542796
122,2025-Q3,55288.940684,2025,Q3,9.054541
123,2025-Q4,52589.611030,2025,Q4,-4.882223


In [11]:
df_GDP.info()

<class 'pandas.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   period      125 non-null    str    
 1   GDP         125 non-null    float64
 2   year        125 non-null    str    
 3   quartal     125 non-null    str    
 4   GDP_growth  124 non-null    float64
dtypes: float64(2), str(3)
memory usage: 6.6 KB


In [12]:

df_DEF=api_to_PandasDataframe("DEFICIT","GR")
df_DEF=clean_df(df_DEF,"DEF")

In [13]:
df_DEF.head()

,period,DEFICIT,year,quartal
0,1999-Q1,-1.919690,1999,Q1
1,1999-Q2,-3.359824,1999,Q2
2,1999-Q3,-3.184237,1999,Q3
3,1999-Q4,-6.024000,1999,Q4
4,2000-Q1,-5.244000,2000,Q1


In [14]:
df_DEF.info()

<class 'pandas.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   period   108 non-null    str    
 1   DEFICIT  108 non-null    float64
 2   year     108 non-null    str    
 3   quartal  108 non-null    str    
dtypes: float64(1), str(3)
memory usage: 4.9 KB


In [15]:
df_HICP=groupby_df(df_HICP,"period","HICP","mean")

In [16]:
df_HICP

,period,HICP
0,1996-Q4,NaN
1,1997-Q1,6.333333
2,1997-Q2,5.566667
3,1997-Q3,5.233333
4,1997-Q4,4.700000
...,...,...
114,2025-Q2,3.166667
115,2025-Q3,2.866667
116,2025-Q4,2.433333
117,2026-Q1,3.133333


In [17]:
merged_df=join_df(df_list=[df_HICP,df_GDP,df_DEF],join_how="inner",primary_key="period")

In [21]:
merged_df

,period,HICP,GDP,GDP_growth,DEFICIT,year,quartal
0,1999-Q1,3.133333,39261.945633,-8.706132,-1.919690,1999,Q1
1,1999-Q2,2.033333,40089.720174,2.108338,-3.359824,1999,Q2
2,1999-Q3,1.433333,43325.428895,8.071168,-3.184237,1999,Q3
3,1999-Q4,2.000000,44611.993661,2.969537,-6.024000,1999,Q4
4,2000-Q1,2.600000,40778.074693,-8.593920,-5.244000,2000,Q1
...,...,...,...,...,...,...,...
103,2024-Q4,3.000000,51297.989876,-5.318786,1.347100,2024,Q4
104,2025-Q1,3.066667,45863.170264,-10.594605,2.393300,2025,Q1
105,2025-Q2,3.166667,50698.430561,10.542796,2.141200,2025,Q2
106,2025-Q3,2.866667,55288.940684,9.054541,2.458900,2025,Q3


In [22]:
merged_df.drop(
    columns=[
        "GDP"
    ],
    inplace=True
)

In [23]:
merged_df

,period,HICP,GDP_growth,DEFICIT,year,quartal
0,1999-Q1,3.133333,-8.706132,-1.919690,1999,Q1
1,1999-Q2,2.033333,2.108338,-3.359824,1999,Q2
2,1999-Q3,1.433333,8.071168,-3.184237,1999,Q3
3,1999-Q4,2.000000,2.969537,-6.024000,1999,Q4
4,2000-Q1,2.600000,-8.593920,-5.244000,2000,Q1
...,...,...,...,...,...,...
103,2024-Q4,3.000000,-5.318786,1.347100,2024,Q4
104,2025-Q1,3.066667,-10.594605,2.393300,2025,Q1
105,2025-Q2,3.166667,10.542796,2.141200,2025,Q2
106,2025-Q3,2.866667,9.054541,2.458900,2025,Q3
